## Abstract

First Data for Passive Data Collection using Smartwatches and GPS from the PREACT Study. 

## Introduction

Treatment personalization is highly discussed to counteract insufficient response rates in psychotherapy. In the quest for criteria allowing informed selection or adaptation, ambulatory assessment data (i.e. EMA, passive sensing)are a key component, as processes happening outside of therapy sessions can be depicted in high temporal and/or spatial resolution.

PREACT is a multicenter prospective-longitudinal study investigating different predictors of non-response (i.e. EEG, fMRI) in around 500 patients undergoing cognitive behavioral therapy for internalizing disorders (https://forschungsgruppe5187.de/de). 

## Methods
Patients can enroll for therapy-accompanying ambulatory assessment. They are provided with a customized study app and a state-of-the-art smartwatch collecting passive data like GPS and heart rate for up to 365 days. In parallel, three 14-day EMA phases (pre-, mid- and post-therapy) cover transdiagnostic (i.e. emotion regulation), contextual and therapy-related aspects.  

Here, we present first results on data compliance and quality for the passive sensing data as well as EMA assessments.


In [1]:
import os
import glob
import pickle
import sys
import re
# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)

import pandas as pd
import datetime as dt
from datetime import date, datetime
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from server_config import datapath, proj_sheet,preprocessed_path, raw_path

today = date.today().strftime("%d%m%Y")
today_day = pd.to_datetime('today').normalize()


df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")


In [2]:
df_monitoring = df_monitoring.copy()
df_monitoring.rename(columns = {"Pseudonym": "customer", "EMA_ID": "ema_id", "Status": "status",
                                "Studienversion":"study_version", "FOR_ID":"for_id", 
                           "Start EMA Baseline": "ema_base_start", "Ende EMA Baseline": "ema_base_end", 
                           "Freischaltung/ Start EMA T20": "ema_t20_start","Ende EMA T20":"ema_t20_end", 
                               "Termin 1. Gespräch": "first_call_date", "Freischaltung/ Start EMA Post":"ema_post_start",
                               "Ende EMA Post":"ema_post_end", "T20=Post":"t20_post" }, inplace=True)

df_monitoring["customer"] = df_monitoring["customer"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(df_monitoring["ema_base_start"], dayfirst=True)
df_monitoring["ema_base_end"] = pd.to_datetime(df_monitoring["ema_base_end"], dayfirst=True)

df_monitoring_short = df_monitoring[["customer", "status", "study_version", "ema_base_start","ema_base_end"]]

### Backup Data 1

In [3]:
# small backup passive data
#file_pattern_back_1 = os.path.join(datapath, 'raw/tiki_backup_files/export_tiki_27052024/"epoch_part*.csv"')
datapath_back = os.path.join(raw_path, "tiki_backup_files/export_tiki_21052024/")
file_pattern_back_1 = os.path.join(datapath_back,"epoch_part*.csv")  # Adjust the path and extension if needed

backup_files = glob.glob(file_pattern_back_1)
file_list = glob.glob(file_pattern_back_1)
file_list.sort()
df_backup_small = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list), ignore_index=True)


In [4]:
df_backup_small["start_end"] = df_backup_small["endTimestamp"] - df_backup_small["startTimestamp"]
df_backup_small["startTimestamp"] = pd.to_datetime(df_backup_small["startTimestamp"],unit='ms')
df_backup_small["endTimestamp"] = pd.to_datetime(df_backup_small["endTimestamp"],unit='ms')

df_backup_small["customer"] = df_backup_small.customer.str.split("@").str.get(0)
df_backup_small["customer"] = df_backup_small["customer"].str[:4]

df_backup_small['startTimestamp'] = df_backup_small['startTimestamp'] + pd.to_timedelta(df_backup_small['timezoneOffset'], unit='m')
df_backup_small['endTimestamp'] = df_backup_small['endTimestamp'] + pd.to_timedelta(df_backup_small['timezoneOffset'], unit='m')

df_backup_small["startTimestamp_day"] = df_backup_small.startTimestamp.dt.normalize()
df_backup_small["startTimestamp_hour"] = df_backup_small.startTimestamp.dt.hour

### Backup Data 2

In [5]:

# Define the pattern for big backup passive data files
file_pattern_back_2 = os.path.join(raw_path, 'tiki_backup_files/tiki_backup_*.csv')

# Use glob to find all matching files
big_backup_files = glob.glob(file_pattern_back_2)

# Define the dtype for columns that are known to be problematic
dtype_spec = {
    'startTimestamp': 'str',  # Load as string initially
    'endTimestamp': 'str'     # Load as string initially
}

# Create a list to hold all the dataframes
all_dfs = []

# Loop over the files and read them with date parsing
for filename in big_backup_files:
    df = pd.read_csv(
        filename,
        dtype=dtype_spec,  # Load timestamps as strings first
        low_memory=False  # Ensure proper memory handling
    )
    
    # Convert the timestamp columns to datetime, ensuring proper parsing of ISO 8601 format
    df['starttimestamp'] = pd.to_datetime(df['startTimestamp'], format='%Y-%m-%dT%H:%M:%S%z', errors='coerce')
    df['endtimestamp'] = pd.to_datetime(df['endTimestamp'], format='%Y-%m-%dT%H:%M:%S%z', errors='coerce')

    
    # Additional processing (e.g., adding date and index columns)
    match = re.match(r'tiki_backup_(\d{4}-\d{2}-\d{2})_(\d+)\.csv', os.path.basename(filename))
    if match:
        date_part = match.group(1)  # Extract the date
        index_part = int(match.group(2))  # Extract the index
        # Add the date and index as new columns to the dataframe
        df['date'] = date_part
        df['index'] = index_part

    all_dfs.append(df)

# Concatenate all dataframes
df_backup_big = pd.concat(all_dfs, ignore_index=True)

# Sort the merged dataframe by 'date' and 'index'

# Optionally, drop the 'date' and 'index' columns if they are no longer needed
df_backup_big.drop(columns=['date', 'index'], inplace=True)


/tmp/ipykernel_2820744/2867167067.py:25: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['starttimestamp'] = pd.to_datetime(df['startTimestamp'], format='%Y-%m-%dT%H:%M:%S%z', errors='coerce')
/tmp/ipykernel_2820744/2867167067.py:26: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['endtimestamp'] = pd.to_datetime(df['endTimestamp'], format='%Y-%m-%dT%H:%M:%S%z', errors='coerce')
/tmp/ipykernel_2820744/2867167067.py:25: FutureWarning: In a fut

In [6]:
# Convert the 'startTimestamp' and 'endTimestamp' columns to datetime objects
df_backup_big['startTimestamp'] = pd.to_datetime(df_backup_big['startTimestamp'], errors='coerce')
df_backup_big['endTimestamp'] = pd.to_datetime(df_backup_big['endTimestamp'], errors='coerce')
# Convert the 'startTimestamp' and 'endTimestamp' columns to datetime objects
df_backup_big['startTimestamp'] = pd.to_datetime(df_backup_big['startTimestamp'], errors='coerce')
df_backup_big['endTimestamp'] = pd.to_datetime(df_backup_big['endTimestamp'], errors='coerce')

# Adjust for timezone offset
df_backup_big['startTimestamp'] = df_backup_big['startTimestamp'] + pd.to_timedelta(df_backup_big['timezoneOffset'], unit='m')
df_backup_big['endTimestamp'] = df_backup_big['endTimestamp'] + pd.to_timedelta(df_backup_big['timezoneOffset'], unit='m')


# Format the datetime objects to the desired format
df_backup_big['startTimestamp'] = df_backup_big['startTimestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_backup_big['endTimestamp'] = df_backup_big['endTimestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

df_backup_big['startTimestamp'] = pd.to_datetime(df_backup_big['startTimestamp'], errors='coerce')
df_backup_big['endTimestamp'] = pd.to_datetime(df_backup_big['endTimestamp'], errors='coerce')

df_backup_big["startTimestamp_day"] = df_backup_big.startTimestamp.dt.normalize()
df_backup_big["startTimestamp_hour"] = df_backup_big.startTimestamp.dt.hour

/tmp/ipykernel_2820744/2271628331.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_backup_big['startTimestamp'] = pd.to_datetime(df_backup_big['startTimestamp'], errors='coerce')
/tmp/ipykernel_2820744/2271628331.py:3: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_backup_big['endTimestamp'] = pd.to_datetime(df_backup_big['endTimestamp'], errors='coerce')


### Backup Data 3

In [7]:
# from january

# small backup passive data
#file_pattern_back_1 = os.path.join(datapath, 'raw/tiki_backup_files/export_tiki_27052024/"epoch_part*.csv"')
datapath_back_3 = os.path.join(raw_path, "tiki_backup_files/export_tiki_20012025/")
file_pattern_back_3 = os.path.join(datapath_back_3,"epoch_part*.csv")  # Adjust the path and extension if needed

backup_files_3 = glob.glob(file_pattern_back_3)
file_list_3 = glob.glob(file_pattern_back_3)
file_list_3.sort()
df_backup_small_3 = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list_3), ignore_index=True)

In [8]:
df_backup_small_3["start_end"] = df_backup_small_3["endTimestamp"] - df_backup_small_3["startTimestamp"]
df_backup_small_3["startTimestamp"] = pd.to_datetime(df_backup_small_3["startTimestamp"],unit='ms')
df_backup_small_3["endTimestamp"] = pd.to_datetime(df_backup_small_3["endTimestamp"],unit='ms')

df_backup_small_3["customer"] = df_backup_small_3.customer.str.split("@").str.get(0)
df_backup_small_3["customer"] = df_backup_small_3["customer"].str[:4]

df_backup_small_3['startTimestamp'] = df_backup_small_3['startTimestamp'] + pd.to_timedelta(df_backup_small_3['timezoneOffset'], unit='m')
df_backup_small_3['endTimestamp'] = df_backup_small_3['endTimestamp'] + pd.to_timedelta(df_backup_small_3['timezoneOffset'], unit='m')

df_backup_small_3["startTimestamp_day"] = df_backup_small_3.startTimestamp.dt.normalize()
df_backup_small_3["startTimestamp_hour"] = df_backup_small_3.startTimestamp.dt.hour

### Merge 3 dataframes

In [9]:
latest_timestamp_big = df_backup_big['startTimestamp'].max()
latest_timestamp_small = df_backup_small['startTimestamp'].max()

# Filter the second dataframe to include only entries after the latest timestamp
df_backup_small_filtered = df_backup_small[df_backup_small['startTimestamp'] > latest_timestamp_big]

In [10]:
df_backup_3_filtered = df_backup_small_3[df_backup_small_3['startTimestamp'] > latest_timestamp_small]

In [ ]:
# Sort the original DataFrames by startTimestamp
df_backup_big_sorted = df_backup_big.sort_values(by='startTimestamp')
df_backup_small_filtered_sorted = df_backup_small_filtered.sort_values(by='startTimestamp')
df_backup_3_filtered_sorted = df_backup_3_filtered.sort_values(by='startTimestamp')

# Concatenate the sorted DataFrames
result_df_final = pd.concat([df_backup_big_sorted, df_backup_small_filtered_sorted], ignore_index=True)
result_df_final = pd.concat([result_df_final, df_backup_3_filtered_sorted], ignore_index=True)


# Optional: Sort the final DataFrame again to ensure everything is in order
result_df_final = result_df_final.sort_values(by='startTimestamp').reset_index(drop=True)


In [ ]:
result_df_final["customer"] = result_df_final.customer.str.split("@").str.get(0)
result_df_final["customer"] = result_df_final["customer"].str[:4]

In [ ]:
print("Minimum timestamp:", result_df_final['startTimestamp'].min())
print("Maximum timestamp:", result_df_final['startTimestamp'].max())


### Export

In [ ]:
result_df_final_merged = result_df_final.merge(df_monitoring_short, on="customer", how="right")

In [ ]:
result_df_final_merged = result_df_final_merged.drop(columns=['valueType', 'createdAt', 'source', 'trustworthiness', 'medicalGrade', 'generation'])

In [ ]:
object_cols = ["booleanValue", "stringValue", "status", "study_version", "customer", "type"] 

In [ ]:
# Fill NaN values with -99 for the specified columns
for col in object_cols:
    result_df_final_merged[col] = result_df_final_merged[col].fillna(-99)

# Convert "booleanValue" to boolean
result_df_final_merged['booleanValue'] = result_df_final_merged['booleanValue'].apply(lambda x: bool(x) if x != -99 else False)

# Convert "stringValue", "status", "study_version" to string using StringDtype
result_df_final_merged['stringValue'] = result_df_final_merged['stringValue'].astype('string')
result_df_final_merged['status'] = result_df_final_merged['status'].astype('string')
result_df_final_merged['study_version'] = result_df_final_merged['study_version'].astype('string')
result_df_final_merged['customer'] = result_df_final_merged['customer'].astype('string')
result_df_final_merged['type'] = result_df_final_merged['type'].astype('string')


In [ ]:
# Calculate memory usage in bytes
memory_usage_bytes = result_df_final.memory_usage(deep=True).sum()

# Convert to megabytes
memory_usage_mb = memory_usage_bytes / (1024 ** 2)

# Convert to gigabytes
memory_usage_gb = memory_usage_bytes / (1024 ** 3)

# Convert to terabytes
memory_usage_tb = memory_usage_bytes / (1024 ** 4)

print(f"Memory usage: {memory_usage_bytes} bytes")
print(f"Memory usage: {memory_usage_mb:.2f} MB")
print(f"Memory usage: {memory_usage_gb:.2f} GB")
print(f"Memory usage: {memory_usage_tb:.2f} TB")


In [ ]:
backup_path = preprocessed_path + "/backup_data_passive_general.feather"

In [ ]:
# Save to HDF5 format
result_df_final_merged.to_feather(backup_path)